# Sales Forecast AI - Dataset Preprocessing

Use this Google Colab notebook to clean sales datasets before uploading them into the Streamlit website.

The notebook accepts flexible column names, maps them into the app schema, fills safe defaults, engineers time-series features, and exports `cleaned_sales_data.csv`.

In [ ]:
import numpy as np
import pandas as pd
from google.colab import files

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)


## 1. Upload Sales Data

Upload a CSV, TSV, TXT, Excel, JSON, JSONL, or Parquet file. If you skip upload, the notebook can load the sample CSV from the GitHub repo.

In [ ]:
USE_GITHUB_SAMPLE = True

if USE_GITHUB_SAMPLE:
    url = 'https://raw.githubusercontent.com/og-harish/streamlit-sale-ml/main/sample_data/sample_sales.csv'
    raw_df = pd.read_csv(url)
    dataset_name = 'sample_sales.csv'
else:
    uploaded = files.upload()
    dataset_name = next(iter(uploaded))
    suffix = dataset_name.lower().rsplit('.', 1)[-1]
    if suffix in {'csv', 'txt'}:
        raw_df = pd.read_csv(dataset_name)
    elif suffix == 'tsv':
        raw_df = pd.read_csv(dataset_name, sep='\t')
    elif suffix in {'xlsx', 'xls'}:
        raw_df = pd.read_excel(dataset_name)
    elif suffix == 'jsonl':
        raw_df = pd.read_json(dataset_name, lines=True)
    elif suffix == 'json':
        raw_df = pd.read_json(dataset_name)
    elif suffix == 'parquet':
        raw_df = pd.read_parquet(dataset_name)
    else:
        raise ValueError(f'Unsupported file type: {suffix}')

print(dataset_name, raw_df.shape)
raw_df.head()

## 2. Auto-Detect Columns

The Streamlit app supports flexible sales files. These aliases map common business column names into the forecast schema.

In [ ]:
COLUMN_ALIASES = {
    'date': ['date', 'order_date', 'sale_date', 'created_at', 'timestamp', 'transaction_date', 'invoice_date', 'purchase_date'],
    'region': ['region', 'state', 'location', 'city', 'area', 'district', 'country', 'market', 'zone'],
    'product_category': ['product_category', 'category', 'product', 'item', 'product_type', 'product_name', 'sku', 'department', 'sub_category'],
    'units_sold': ['units_sold', 'quantity', 'qty', 'units', 'sales_count', 'items_sold', 'order_qty', 'count'],
    'revenue': ['revenue', 'sales', 'amount', 'total_price', 'net_sales', 'gross_sales', 'order_value', 'gmv', 'subtotal'],
    'unit_price': ['unit_price', 'selling_price', 'price', 'rate', 'mrp', 'item_price', 'list_price'],
    'discount_pct': ['discount_pct', 'discount', 'discount_percent', 'offer', 'promo_discount', 'coupon_discount'],
    'customer_reviews': ['customer_reviews', 'review', 'feedback', 'comment', 'customer_text', 'customer_review', 'remarks', 'rating_text'],
}

def normalize_column_name(name):
    return str(name).strip().lower().replace(' ', '_').replace('-', '_')

def auto_detect_column_mapping(df):
    normalized = {normalize_column_name(col): col for col in df.columns}
    mapping = {}
    for field, aliases in COLUMN_ALIASES.items():
        mapping[field] = next((normalized[normalize_column_name(alias)] for alias in aliases if normalize_column_name(alias) in normalized), None)
    return mapping

column_mapping = auto_detect_column_mapping(raw_df)
pd.DataFrame([{'field': k, 'mapped_column': v or 'fallback'} for k, v in column_mapping.items()])

## 3. Preprocess Into Forecast Schema

The cleaned dataset uses the website schema: `date`, `region`, `product_category`, `units_sold`, `revenue`, `discount_pct`, and `customer_reviews`.

In [ ]:
def preprocess_data(df, mapping):
    source = df.drop_duplicates().copy()
    clean = pd.DataFrame(index=source.index)

    if mapping.get('date'):
        clean['date'] = pd.to_datetime(source[mapping['date']], errors='coerce')
        fallback_date = clean['date'].dropna().min() if clean['date'].notna().any() else pd.Timestamp('2026-01-01')
        clean['date'] = clean['date'].fillna(fallback_date)
    else:
        clean['date'] = pd.date_range('2026-01-01', periods=len(clean), freq='D')

    clean['region'] = source[mapping['region']].fillna('Overall').astype(str).str.strip() if mapping.get('region') else 'Overall'
    clean['product_category'] = source[mapping['product_category']].fillna('General').astype(str).str.strip() if mapping.get('product_category') else 'General'
    clean['customer_reviews'] = source[mapping['customer_reviews']].fillna('').astype(str) if mapping.get('customer_reviews') else 'No review text available'
    clean['units_sold'] = pd.to_numeric(source[mapping['units_sold']], errors='coerce') if mapping.get('units_sold') else np.nan
    clean['revenue'] = pd.to_numeric(source[mapping['revenue']], errors='coerce') if mapping.get('revenue') else np.nan

    if clean['revenue'].isna().all() and mapping.get('unit_price') and mapping.get('units_sold'):
        unit_price = pd.to_numeric(source[mapping['unit_price']], errors='coerce')
        clean['revenue'] = clean['units_sold'] * unit_price

    if clean['units_sold'].isna().all():
        clean['units_sold'] = np.maximum(1, np.round(clean['revenue'].fillna(clean['revenue'].median()) / max(1, clean['revenue'].median() / 20)))
    clean['units_sold'] = clean['units_sold'].fillna(clean['units_sold'].median()).clip(lower=0)
    clean['revenue'] = clean['revenue'].fillna(clean['units_sold'] * max(1, clean['revenue'].median() / max(1, clean['units_sold'].median()))).clip(lower=0)
    clean['discount_pct'] = pd.to_numeric(source[mapping['discount_pct']], errors='coerce') if mapping.get('discount_pct') else 0
    clean['discount_pct'] = clean['discount_pct'].fillna(0).clip(lower=0, upper=95)

    clean = clean.sort_values('date').reset_index(drop=True)
    clean['month'] = clean['date'].dt.month
    clean['year'] = clean['date'].dt.year
    clean['day_of_week'] = clean['date'].dt.dayofweek
    clean['day_of_month'] = clean['date'].dt.day
    clean['quarter'] = clean['date'].dt.quarter
    clean['is_weekend'] = clean['day_of_week'].isin([5, 6]).astype(int)
    clean['lag_7'] = clean['revenue'].shift(7).fillna(clean['revenue'].median())
    clean['lag_30'] = clean['revenue'].shift(30).fillna(clean['revenue'].median())
    clean['rolling_mean_7'] = clean['revenue'].rolling(7, min_periods=1).mean()
    clean['rolling_mean_30'] = clean['revenue'].rolling(30, min_periods=1).mean()
    clean['rolling_std_7'] = clean['revenue'].rolling(7, min_periods=2).std().fillna(0)
    return clean

clean_df = preprocess_data(raw_df, column_mapping)
clean_df.head()

## 4. Quality Checks

In [ ]:
summary = {
    'rows': len(clean_df),
    'date_min': clean_df['date'].min(),
    'date_max': clean_df['date'].max(),
    'total_revenue': clean_df['revenue'].sum(),
    'total_units': clean_df['units_sold'].sum(),
    'regions': clean_df['region'].nunique(),
    'categories': clean_df['product_category'].nunique(),
    'missing_values': int(clean_df.isna().sum().sum()),
}
summary

## 5. Export For Streamlit Website

In [ ]:
output_file = 'cleaned_sales_data.csv'
clean_df.to_csv(output_file, index=False)
files.download(output_file)
print(f'Exported {output_file} with {len(clean_df):,} rows.')